This program is part of pyHNC, copyright (c) 2023 Patrick B Warren (STFC).
Additional modifications copyright (c) 2025 Joshua F Robinson (STFC).
Email: patrick.warren{at}stfc.ac.uk.

This program is free software: you can redistribute it and/or modify
it under the terms of the GNU General Public License as published by
the Free Software Foundation, either version 3 of the License, or
(at your option) any later version.

This program is distributed in the hope that it will be useful, but
WITHOUT ANY WARRANTY; without even the implied warranty of
MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the GNU
General Public License for more details.

You should have received a copy of the GNU General Public License
along with this program.  If not, see
<http://www.gnu.org/licenses/>.

Demonstrate the capabilities of the HNC package for solving DPD
potentials, comparing with SunlightHNC if requested, and plotting
the pair distribution function and the structure factor too.  For
details here see also the SunlightHNC documentation.

For standard DPD at $A = 25$ and $ρ = 3$, we have the following table

```
          ∆t = 0.02   ∆t = 0.01   Monte-Carlo  HNC   deviation
pressure  23.73±0.02  23.69±0.02  23.65±0.02   23.564  (0.4%)
energy    13.66±0.02  13.64±0.02  13.63±0.02   13.762  (1.0%)
mu^ex     12.14±0.02  12.16±0.02  12.25±0.10   12.170  (0.7%)
```


The first two columns are from dynamic simulations.  The excess
chemical potential (final row) is measured by Widom insertion.  The
HNC results from the present code are in agreement with those from
SunlightHNC to at least the indicated number of decimals.  The
deviation is between HNC and simulation results.

Data is from a forthcoming publication on osmotic pressure in DPD.

## Standard modules

In [ ]:
import os
import numpy as np, matplotlib.pyplot as plt
import pandas as pd
from numpy import pi as π
from scipy.integrate import simpson
from pyhnc import *

## Global parameters

In [ ]:
N = 2**13
L = 250
grid = Grid(L, N)
r, q = grid.r, grid.q

verbose = False

alpha = 0.5
niters = 1000
tol = 1e-12
solvent = Solver(grid, alpha=alpha, niters=niters, tol=tol)
solvent_rpa = RandomPhaseApproximationSolver(grid, alpha=alpha, niters=niters, tol=tol)

print(grid)
print(solvent)

# 1. Solve for pure solvent

Define interaction parameters:

In [ ]:
# Parameters for solvent-solvent interactions
A00 = 25
ρ0 = 3.0
φ0 = potentials.DPD(A00)

Solve for $h(r)$ iteratively with HNC closure:

In [ ]:
sol0 = solvent.solve(φ0, ρ0, monitor=verbose)
sol0_rpa = solvent_rpa.solve(φ0, ρ0, monitor=verbose)

φ0q = grid.fourier_bessel_forward(φ0.potential(r))
h00q_rpa_pure = -φ0q / (1 + ρ0 * φ0q)
h00_rpa_pure = grid.fourier_bessel_backward(h00q_rpa_pure)

plt.figure(figsize=(3.375, 3))
plt.plot(r, sol0.g, label='HNC')
plt.plot(r, sol0_rpa.g, '--', label='RPA/test')
plt.plot(r, 1+h00_rpa_pure, ':', label='RPA original')
plt.legend(loc='best')
plt.xlabel('$r$')
plt.ylabel('$g(r)$')
plt.xlim([0, 3])
plt.ylim([0, 1.5])
plt.show()

Calculate thermodynamic quantities using the total correlation function $h(r)$.

For the integrals here, see Eqs. (2.5.20) and (2.5.22) in Hansen & McDonald, "Theory of Simple Liquids" (3rd edition): for the (excess) energy density,
$$
e \equiv \frac{U^\mathrm{ex}}{V} = 2\pi\rho^2 \int_0^\infty \mathrm{d}r \, r^2 \phi(r) g(r)
$$
and virial pressure,
$$
p = \rho + \frac{2\pi\rho^2}{3} \int_0^\infty \mathrm{d}r \, r^3 f(r) g(r)
$$
where $f(r) = −\mathrm{d}\phi/\mathrm{d}r$ is the force. Note that we have assumed $\beta = 1$ in our expressions, so energy is given in units of $k_\mathrm{B} T$. An integration by parts shows that the mean-field contributions, being these with g(r) = 1, are the same.

Here specifically the mean-field contributions are
$$
\frac{2\pi \rho^3}{3} \int_0^\infty \mathrm{d}r \, r^3 f(r) = A \int_0^1 \mathrm{d}r \, r^3 (1−r) = \frac{\pi A \rho^3}{30} \,.
$$

In [ ]:
# e_mf = p_mf = π*A00*ρ0**2/30

# e_xc = 2*π*ρ0**2 * np.trapz(r**2*φ0*h00, dx=Δr)
# e_ex = e_mf + e_xc
# e = 3*ρ0/2 + e_ex

# p_xc = 2*π*ρ0**2/3 * np.trapz(r**3*f0*h00, dx=Δr)
# p_ex = p_mf + p_xc
# p = ρ0 + p_ex

Routes to chemical potential need a write up.

In [ ]:
class ExcessChemicalPotential:
    """Chemical potential by different routes."""

    @classmethod
    def test_particle(cls, sol):
        """Chemical potential/test particle route with HNC closure."""
        h, c, r = sol.h, sol.c, sol.r
        try: rho = sol.solvent.rho
        except: rho = sol.rho
        return 4*π*rho * simpson(r**2*(h*(h-c)/2 - c), r)

    @classmethod
    def compressibility(cls, sol):
        """Generic compressibility route."""
        h, c, r = sol.h, sol.c, sol.r
        try: rho = sol.solvent.rho
        except: rho = sol.rho
        return -4*π*rho * simpson(r**2*c, r)

μ_ex = ExcessChemicalPotential.test_particle(sol0)
μ = np.log(sol0.rho) + μ_ex
print(f'  test particle: μ={μ:.4f} μ_ex={μ_ex:.4f} μ_id={μ-μ_ex:.4f}')
μ_ex = ExcessChemicalPotential.compressibility(sol0)
μ = np.log(sol0.rho) + μ_ex
print(f'compressibility: μ={μ:.4f} μ_ex={μ_ex:.4f} μ_id={μ-μ_ex:.4f}')

# 2. Introduce solute

We need to use a different solver that takes into account that this is now a binary mixture where the second species (the solute) is infinitely dilute. We need to pass the previously obtained static properties of the solvent.

In [ ]:
solute = SoluteSolver(sol0, alpha=alpha, niters=niters, tol=tol)
solute_rpa = SoluteTestParticleRPA(sol0_rpa, alpha=alpha, niters=niters, tol=tol)

RISM approach:

In [ ]:
l = 0
omega12 = np.sinc(grid.q * l / np.pi)

A01 = 25
φ01 = potentials.DPD(A01)
dimer = SoluteHomodimerSolver(sol0, omega12)
dimer.solve(φ01)
solute.solve(φ01)

# alternative calculation: h_{01} rather than h_{10}. should be equal.
# trial_hq = (1 + omega12) * dimer.cq / (1 - sol0.rho * sol0.cq)
# plt.plot(grid.q, trial_hq - dimer.hq)
# plt.xlim([0, 25])

plt.plot(grid.r, dimer.g, label='RISM $g_{01}$')
plt.plot(grid.r, solute.g, label='HNC solute $g_{01}$')
plt.plot(grid.r, sol0.g, '--', label='HNC $g_{00}$')
plt.legend(loc='best')
plt.xlabel(r'$r/\sigma$')
plt.ylabel('$g(r)$')
plt.xlim([0, 5])
plt.show()

Show how the distribution function $g_{01}$ between solvent (species 0) and solute (species 1) varies with DPD interaction parameter $A_{01}$. We can also infer $g_{11}$ directly from $g_{01}$ and thereby calculate the potential of mean force between two solute particles:
$$
- \ln{g_{11}(r)} = \beta \phi_{11}(r) + \beta W_{11}(r)\,.
$$
Here $\beta W_{11}$ is the depletion potential between solutes.

In [ ]:
# Monte-Carlo data for comparison.
data_file = '../data/dimer_mu_mc1.ods'
muref_data = pd.read_excel(data_file, sheet_name='muref_all.dat')
cut = (muref_data['quantity'] == 'mu') & (muref_data['njobs'] == 100)
muex_mon = muref_data[cut].set_index('A01')['value'] # a pandas Series indexed by A01
dimer_data = pd.read_excel(data_file, sheet_name='dimer_all.dat')
dimer_data2 = pd.read_csv('../data/dimer_mu_mc2.csv')

fig1 = plt.figure(figsize=(3.375, 2.5))
ax1 = plt.gca()
fig2 = plt.figure(figsize=(3.375, 3.375))
ax2 = plt.gca()

ax1.plot(r, sol0.g, label=(r'$g_{00}(r; A_{00}=' + f'{A00})$'))

# for A01 in np.arange(A00, 151, 25):
A_range = np.arange(5, 26, 5)
# A_range = np.concatenate([A_range, [50]])
first = True
line_plots = []
color_plots = []
for A01 in np.flipud(A_range):
    φ01 = potentials.DPD(A01)
    sol1 = solute.solve(φ01, monitor=verbose)
    sol2 = solute_rpa.solve(φ01, monitor=verbose)
    pl, = ax1.plot(r, sol1.g, '--', label=(r'$g_{01}(r; A_{01}=' + f'{A01})$'))
    ax1.plot(r, sol2.g, ':', c=pl.get_color())

    depletion11 = - ρ0 * grid.fourier_bessel_backward(sol1.cq * sol1.hq)
    pl1, = ax2.plot(r, depletion11, lw=0.5, label='HNC')
    depletion11_rpa = - ρ0 * grid.fourier_bessel_backward(sol2.cq * sol2.hq)
    pl2, = ax2.plot(r, depletion11_rpa, '-.', lw=0.5, c=pl1.get_color(), label=r'TP-RPA Eq.\ (21)')

    vq = grid.fourier_bessel_forward(φ01(grid.r))
    depletion11_rpa2 = ρ0 * grid.fourier_bessel_backward(vq * sol2.hq)
    pl3, = ax2.plot(r, depletion11_rpa2, ls=(0, (3,3,1,1,1,3)),
                    lw=0.5, c=pl1.get_color(), label=r'TP-RPA Eq.\ (20)')

    # Equivalent calculation should be on top of other lines.
    # psi1q = h01q / (1 + ρ0*h00q)**0.5
    # depletion11 = - ρ0 * grid.fourier_bessel_backward(psi1q**2)
    # ax2.plot(r, depletion11, '--', c=pl.get_color())

    # Work out the expected value at g(r=0) if the theory were self-consistent.
    φ01_eff = potentials.DPD(2*A01)
    sol2 = solute.solve(φ01_eff, monitor=verbose)
    μ_ex_monomer = ExcessChemicalPotential.test_particle(sol1)
    μ_ex_eff = ExcessChemicalPotential.test_particle(sol2)
    dμ = μ_ex_eff - 2*μ_ex_monomer
    try:
        xmax = {25: 0.75, 20: 0.7, 15: 0.85, 10: 1.025}
        xmax = xmax[A01]
    except Exception as e:
        xmax = 1.025
    x = np.linspace(0, xmax, 3)
    ax2.plot(x, dμ * np.ones_like(x), ls=(0, (5, 10)), c=pl1.get_color())
    if A01 > 6:
        if first: label = rf'$\mu_1^\text{{ex}}(A_{{01}}={2*A01})$'
        else: label = rf'$\mu_1^\text{{ex}}({2*A01})$'
        ax2.text(x[-1], dμ + 0.05, label, c=pl1.get_color(),
                 ha='right', va='bottom', fontsize=6)
    else:
        ax2.text(x[-1], dμ, rf'$\mu_1^\text{{ex}}({2*A01})$', c=pl1.get_color(),
                 ha='left', va='center', fontsize=6)
    # ax2.axhline(y=dμ, ls='--', c=pl.get_color())

    l_rism = np.linspace(0, 2, 51)
    μ_ex_rism = np.empty_like(l_rism)
    for i,l in enumerate(l_rism):
        omega12 = np.sinc(grid.q * l / np.pi)
        φ01 = potentials.DPD(A01)
        dimer = SoluteHomodimerSolver(sol0, omega12)
        dimer.solve(φ01)
        μ_ex_rism[i] = 2*dimer.excess_chemical_potential
    W_rism = μ_ex_rism - 2*μ_ex_monomer
    pl4, = ax2.plot(l_rism, W_rism, ':', lw=0.5, c=pl1.get_color(), label='RISM')

    fit = dμ * (1+2*r)*(1-r)**2
    fit[r > 1] = 0.
    pl5, = ax2.plot(r, fit, '--', lw=0.5,
                    label='$(1 + 2r)(1 - r)^2$', c=pl1.get_color())

    try:
        cut = (dimer_data['quantity'] == 'mu') & (dimer_data['A01'] == A01)
        muex_dim = dimer_data[cut][['dlength', 'value']].groupby('dlength').mean()['value']
        # muex_err = dimer_data[cut][['dlength', 'error']].groupby('dlength').mean()['error'].values
        # plt.errorbar(muex_dim.index, muex_dim-2*muex_mon[A01], muex_err, c=pl.get_color())#, 'o', mfc='None', ms=1.5, c=pl.get_color())
        plc, = ax2.plot(muex_dim.index, muex_dim-2*muex_mon[A01], 'o',
                        mfc='None', c=pl1.get_color(), label=(r'$A_{01}=' + f'{A01}$'))
    except: pass

    try:
        x = dimer_data2['r'].values
        muex_dim = dimer_data2[f'Aij={A01} L=50'].values
        ax2.plot(x, muex_dim - 2*muex_mon[A01], '^', c=pl1.get_color())
    except: pass

    if first:
        line_plots += [pl1, pl2, pl3, pl4, pl5]
    first = False
    color_plots += [plc]

for ax in [ax1, ax2]:
    ax.set_xlabel('$r$')
    ax.set_xlim([0, 2])
ax1.legend(loc='best', fontsize=8)
line_legend = ax2.legend(handles=line_plots, loc='right', fontsize=8)
ax2.legend(handles=color_plots, loc='lower right', fontsize=8, ncol=3, columnspacing=0.5)
ax2.add_artist(line_legend)
# ax2.set_ylim([-6.5, 0.1])

ax1.set_ylabel('$g(r)$')
ax2.set_ylabel(r'$\beta W_{11}(r)$')

fig1.show()
fig2.savefig('W11.pdf')
fig2.show()

Compare without removing monomer contribution:

In [ ]:
plt.figure(figsize=(3.375, 2.5))
ax = plt.gca()

# for A01 in np.arange(A00, 151, 25):
for A01 in [10, 25, 50]:
    φ01 = potentials.DPD(A01)
    sol1 = solute.solve(φ01, monitor=verbose)
    W11 = - ρ0 * grid.fourier_bessel_backward(sol1.cq * sol1.hq)

    μ_ex_monomer = ExcessChemicalPotential.test_particle(sol1)
    pl, = ax.plot(r, W11 + 2*μ_ex_monomer, label=(r'$A_{01}=' + f'{A01}$'))
    ax.plot(dimer_data2['r'], dimer_data2[f'Aij={A01} L=50'], 'o', c=pl.get_color(), mfc='None')


ax.legend(loc='best', fontsize=8, ncol=3, columnspacing=2)
ax.set_xlabel('$r$')
ax.set_xlim([0, 5])
ax.set_ylim([8, 45])
ax.set_ylabel(r'$W_{11}(r) + 2 \mu_\mathrm{ex}$')
plt.savefig('W11-simulation.pdf')
plt.show()

Look at the scaling of the error with box size:

In [ ]:
plt.figure(figsize=(3.375, 2.5))
L = np.array([10, 30, 50])

# for A01 in np.arange(A00, 151, 25):
for A01 in [10, 25, 50]:
    φ01 = potentials.DPD(A01)
    sol1 = solute.solve(φ01, monitor=verbose)
    μ_ex_monomer = ExcessChemicalPotential.test_particle(sol1)

    W = []
    for l in L:
        W += [dimer_data2[f'Aij={A01} L={l}'].values[-1] / (2*μ_ex_monomer)]
    W = np.array(W)
    plt.plot(1/L**0.5, np.abs(W-1), 'o', mfc='None', label=f'{A01}')

plt.legend(title='$A_{01}$')
plt.xlim([0, 0.4])
# plt.ylim([0.98, 1.1])
plt.xlabel(r'$L^{-1/2}$')
plt.ylabel(r'$\mu_\text{dim}^\text{ex}(r=5) / 2\mu^\text{ex} - 1$')

plt.show()

Calculate excess chemical potentials for solutes with varying solvent-solute interaction parameters $A_{01}$:

In [ ]:
A01 = np.flipud(np.arange(0, 101, 1))
μ_ex_hnc_test = np.empty(len(A01))
μ_ex_hnc_comp = np.empty(len(A01))

for i, A in enumerate(A01[:-1]):
    φ01 = potentials.DPD(A)
    sol1 = solute.solve(φ01, monitor=verbose)
    μ_ex_hnc_test[i] = ExcessChemicalPotential.test_particle(sol1)
    μ_ex_hnc_comp[i] = ExcessChemicalPotential.compressibility(sol1)

μ_ex_hnc_test[-1] = 0.
μ_ex_hnc_comp[-1] = 0.

In [ ]:
μ_ex_rpa_test = np.empty(len(A01))
μ_ex_rpa_comp = np.empty(len(A01))

for i, A in enumerate(A01[:-1]):
    φ01 = potentials.DPD(A)
    sol1 = solute_rpa.solve(φ01, monitor=verbose)
    μ_ex_rpa_test[i] = ExcessChemicalPotential.test_particle(sol1)
    μ_ex_rpa_comp[i] = ExcessChemicalPotential.compressibility(sol1)

μ_ex_rpa_test[-1] = 0.
μ_ex_rpa_comp[-1] = 0.

Plot the previously calculated chemical potentials:
digitised from Fig 1 of Hendrikse *et al.*, PCCP **27**, 1554-66 (2025).


In [ ]:
# fig, (ax, ax2) = plt.subplots(nrows=2, figsize=(3.375, 3.375))
plt.figure(figsize=(3.375, 2.1))
ax = plt.gca()

pl, = ax.plot(A01/A00, μ_ex_hnc_test, '-', lw=0.5, label='HNC/test particle')
# ax.plot(A01/A00, μ_ex_hnc_comp, '--', lw=0.5, c=pl.get_color(), label='HNC/compressibility')

pl2, = ax.plot(A01/A00, μ_ex_rpa_test, '-', lw=0.5, label='RPA/test particle')
# ax.plot(A01/A00, μ_ex_rpa_comp, '--', lw=0.5, c=pl2.get_color(), label='RPA/compressibility (1)')
μ_rpa = ρ0 * np.pi * A01 / 15
ax.plot(A01/A00, μ_rpa, ':', lw=0.5, c=pl2.get_color(), label='RPA/compressibility')

A01_ref, μ_ex_ref = np.genfromtxt('../data/hendrikse2025_mu_dpd.csv', delimiter=',', comments='#').T
ax.plot(A01_ref/A00, μ_ex_ref, 'o', c=pl.get_color(),
         mfc='None', label='Hendrikse (2025)', zorder=10)

# # data_file = '../data/dimer_mu_mc1.ods'
# # muref_data = pd.read_excel(data_file, sheet_name='muref_all.dat')
# cut = (muref_data['quantity'] == 'mu') & (muref_data['njobs'] == 100)
# μ_ex_ref = muref_data[cut].set_index('A01')['value']
# μ_ex_err_ref = muref_data[cut].set_index('A01')['error']
# A01_ref, μ_ex_ref = μ_ex_ref.index[1:].values, μ_ex_ref[1:]
# ax.plot(A01_ref/A00, μ_ex_ref, 'o', c=pl.get_color(),
#         mfc='None', label='MC + Widom', zorder=10)

ax.set_xlabel(r'$A_{01} / A_{00}$')
ax.set_ylabel(r'$\beta \mu_1^\mathrm{ex}$')
ax.legend(loc='best')
ax.set_xlim([0, np.max(A01)/A00])
ax.set_ylim([0, 30])

plt.savefig('monomer-mu.pdf')
plt.show()

Look at convergence towards mean-field behaviour:

In [ ]:
# A00 = 25
# ρ0 = 3.0
# φ0 = potentials.DPD(A00)
A01 = np.flipud(np.arange(0, 1000, 5))
μ_ex = np.zeros(len(A01))

plt.figure()
plt.xlim([0, 1000])
plt.ylim([0, 35])
ax = plt.gca()
trans = ax.transData.transform
inv = ax.transData.inverted().transform

first = True
for rho0 in np.linspace(1, 10, 10):
    solvent_sol = solvent.solve(φ0, rho0, monitor=verbose)
    solute_solver = SoluteSolver(solvent_sol, alpha=alpha, niters=niters, tol=tol)

    for i, A in enumerate(A01[:-1]):
        φ01 = potentials.DPD(A)
        sol1 = solute_solver.solve(φ01, monitor=verbose)
        μ_ex[i] = ExcessChemicalPotential.test_particle(sol1)

    label = rf'{rho0:.1f}'
    # if first: label = rf'$\rho_0={label}$'
    # first = False
    label = rf'$\rho_0={label}$'
    pl, = plt.plot(A01, μ_ex / rho0)
    x = 800
    index = np.where(A01 == x)[0][0]
    y, y1, y2 = [μ_ex[i] / rho0 for i in [index, index-1, index+1]]
    x1, x2 = [A01[i] for i in [index-1, index+1]]
    x, y = trans((x, y))
    x1, y1 = trans((x1, y1))
    x2, y2 = trans((x2, y2))
    dydx = (y2 - y1) / (x2 - x1)
    angle = np.degrees(np.arctan(dydx))
    normal = np.array([-dydx, 1]) / np.sqrt(1 + dydx**2)
    offset = 12.5
    x, y = x + offset*normal[0], y + offset*normal[1]
    x, y = inv((x, y))
    plt.text(x, y, label, rotation=angle, rotation_mode='anchor',
             ha='center', va='center', fontsize=8, c=pl.get_color())

    # μ_rpa = rho0 * np.pi * A01 / 15
    # plt.plot(A01/A00, μ_rpa, ':', lw=0.5, c=pl.get_color())

# solute_solver = SoluteTestParticleRPA(solvent_sol, grid, alpha=alpha, niters=niters, tol=tol)
# μ_ex2 = np.zeros(len(A01))
φ01 = potentials.DPD(1)
A01 = np.arange(0, 200, 5)
μ_ex = simpson(4*np.pi*r**2*φ01(r), r) * A01
#     sol1 = solute_solver.solve(φ01, monitor=verbose)
#     μ_ex[i] = ExcessChemicalPotential.test_particle(sol1)
#     μ_ex2[i] = ExcessChemicalPotential.compressibility(sol1)
#     # sol1 = solute_rpa.solve(φ01, monitor=verbose)
#     # μ_ex_rpa_test[i] = ExcessChemicalPotential.test_particle(sol1)
plt.plot(A01, μ_ex, 'k:', lw=1., label='RPA')

x = 100
index = np.where(A01 == x)[0][0]
y, y1, y2 = [μ_ex[i] for i in [index, index-1, index+1]]
x1, x2 = [A01[i] for i in [index-1, index+1]]
x, y = trans((x, y))
x1, y1 = trans((x1, y1))
x2, y2 = trans((x2, y2))
dydx = (y2 - y1) / (x2 - x1)
angle = np.degrees(np.arctan(dydx))
normal = np.array([-dydx, 1]) / np.sqrt(1 + dydx**2)
offset = 15
x, y = x + offset*normal[0], y + offset*normal[1]
x, y = inv((x, y))
plt.text(x, y, 'RPA', rotation=angle, rotation_mode='anchor',
         ha='center', va='center', fontsize=8)
# plt.plot(A01, μ_ex2, 'k--', zorder=-10)

# plt.ylim([0, 60])

# plt.legend(loc='best')
plt.xlabel(r'$A_{01} / A_{00}$')
plt.ylabel(r'$\beta \mu^\mathrm{ex} / \rho$')
plt.savefig('monomer-mu-b.pdf')
plt.show()

# 3. Partition coefficients of dimers

Taking the depletion potential from the previous section we can integrate over all configurations of a dimer to obtain its chemical potential:

In [ ]:
def bond_potential(r, l0=0.5, k=150):
    """Spring force between bonded atoms in a molecule."""
    return k * (r - l0)**2

from scipy.integrate import simpson

def dimer_excess_chemical_potential(A1, A2, l0=0.5, k=150):
    """Excess chemical potential of a non-rigid dimer is found by integrating
    the potential of mean force over all configurations of the two beads.

    Args:
        A1: interaction strength of bead 1 with the solvent.
        A2: interaction strength of bead 2 with the solvent.
        l0: equilbrium length of dimer in a vacuum.
        k: interaction strength between the two beads (a spring constant).
    Returns:
        Excess chemical potential in units of kT.
    """
    φ01, φ02 = [potentials.DPD(A) for A in [A1, A2]]
    sol1 = solute.solve(φ01, monitor=verbose)
    sol2 = solute.solve(φ02, monitor=verbose)

    # Depletion contribution to potential of mean force
    psi1q, psi2q = [sol.hq / (1 + ρ0*sol0.hq)**0.5 for sol in [sol1, sol2]]
    depletion12 = - ρ0 * grid.fourier_bessel_backward(psi1q*psi2q)

    v12 = bond_potential(r, l0, k)
    φ12 = v12 + depletion12
    return np.log(simpson(4*np.pi * r**2 * np.exp(-φ12), r))

A1 = 25
A2 = np.arange(5, 76, 5)
μ_ex_dimer = [dimer_excess_chemical_potential(A1, A) for A in A2]
plt.plot(A2, μ_ex_dimer)
plt.xlabel('$A_{02}$')
plt.ylabel(r'$\beta \mu_\mathrm{dimer}^\mathrm{ex}$')
plt.title(r'$A_{01} = 25$')
plt.show()


The partition coefficient is obtained from the ratio of excess chemical potentials in two solvents $\alpha$ and $\beta$:
$$
\ln{K_{ij}^{\alpha\beta}}
\equiv
\ln{\left( \frac{c_{ij}^\alpha}{c_{ij}^\beta} \right)}
=
\beta \mu_{ij,\alpha}^\text{ex} - \beta \mu_{ij,\beta}^\text{ex}\,.
$$

In [ ]:
def monomer_partition_coefficient(Aα1, Aβ1):
    """Partition coefficient of a monomer for two reference solvents.
    Args:
        Aα1: interaction coefficient between solute and first solvent.
        Aβ1: interaction coefficient between solute and second solvent.
    Returns:
        Difference between excess chemical potentials in units of kT.
    """
    φα1, φβ1 = [potentials.DPD(A) for A in [Aα1, Aβ1]]

    sol1 = solute.solve(φα1, monitor=verbose)
    μ_ex_α = ExcessChemicalPotential.test_particle(sol1)

    sol2 = solute.solve(φβ1, monitor=verbose)
    μ_ex_β = ExcessChemicalPotential.test_particle(sol2)

    return μ_ex_β - μ_ex_α

def dimer_partition_coefficient(A1, A2, l0=0.5, k=150):
    """Partition coefficient of a dimer for two reference solvents.
    Args:
        A1: interaction coefficients of first bead with each solvent.
        A2: interaction coefficients of second bead with each solvent.
        l0: equilbrium length of dimer in a vacuum.
        k: interaction strength between the two beads (a spring constant).
    Returns:
        Difference between excess chemical potentials in units of kT.
    """

    K1 = monomer_partition_coefficient(*A1)
    K2 = monomer_partition_coefficient(*A2)

    Aα1, Aβ1 = A1
    Aα2, Aβ2 = A2
    μ_ex_α = dimer_excess_chemical_potential(Aα1, Aα2, l0, k)
    μ_ex_β = dimer_excess_chemical_potential(Aβ1, Aβ2, l0, k)

    return K1 + K2 + μ_ex_α - μ_ex_β

A1 = [25, 25]
A2 = [100, 25]
K1 = monomer_partition_coefficient(*A1)
K2 = monomer_partition_coefficient(*A2)

l0 = np.linspace(0, 3, 100)
logP = [dimer_partition_coefficient(A1, A2, ll) for ll in l0]

plt.plot(l0, logP)
plt.axhline(y=(K1+K2), ls='--', lw=0.5)
plt.xlabel('$l_0$')
plt.ylabel(r'$\ln{K_{12}}$')
plt.title(r'$A_{\alpha 1} = 25, A_{\alpha 2} = 100, A_{\beta 1} = A_{\beta 2} = 25$')
plt.show()